# XGBoost GridSearchCV

In [1]:
import os
import sys
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath(os.path.join('..')))


### Memuat dan Split Data

In [2]:
processed_file = os.path.join("..", "data", "processed", "ihsg_processed_features.csv")
if not os.path.exists(processed_file):
    raise FileNotFoundError(f"File {processed_file} tidak ditemukan. Jalankan Data_Preparation.ipynb dulu!")
    
df = pd.read_csv(processed_file, index_col=0, parse_dates=True)
df = df.sort_index()

feature_cols = ['EMA_9', 'SMA_5', 'SMA_15', 'SMA_30', 'RSI', 'MACD', 'MACD_signal']
target_col = 'Target_Close'

X = df[feature_cols]
y = df[target_col]

total_rows = len(df)
train_end = int(total_rows * 0.70)
val_end = train_end + int(total_rows * 0.15)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]
print(f"Data Shapes - Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


Data Shapes - Train: (2695, 7), Val: (577, 7), Test: (579, 7)


### Training Model dengan GridSearchCV

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error

param_grid = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.001, 0.005, 0.01, 0.05],
    'max_depth': [8, 10, 12, 15],
    'gamma': [0.001, 0.005, 0.01, 0.02],
    'random_state': [42]
}

grid = ParameterGrid(param_grid)
print(f"Total kombinasi parameter yang akan dievaluasi: {len(grid)}")

best_val_mse = float('inf')
best_params = None

start_time = time.time()
for i, params in enumerate(grid):
    model = XGBRegressor(**params, n_jobs=-1)
    model.fit(X_train, y_train)
    val_preds = model.predict(X_val)
    val_mse = mean_squared_error(y_val, val_preds)
    if val_mse < best_val_mse:
        best_val_mse = val_mse
        best_params = params
execution_time = time.time() - start_time
print(f"Waktu Eksekusi: {execution_time:.2f} detik")
print(f"Parameter Terbaik: {best_params}")

best_model = XGBRegressor(**best_params, n_jobs=-1)
best_model.fit(X_train, y_train)

test_preds = best_model.predict(X_test)
test_mse = mean_squared_error(y_test, test_preds)
test_rmse = np.sqrt(test_mse)
print(f"Hasil Test Set -> MSE: {test_mse:.4f} | RMSE: {test_rmse:.4f}")


Total kombinasi parameter yang akan dievaluasi: 256


### Evaluasi & Percentage Error

In [ ]:
actual = y_test.values
percentage_errors = np.abs((test_preds - actual) / actual) * 100
mape = np.mean(percentage_errors)
print(f"MAPE (Rata-rata % Error): {mape:.4f}%")

# Simpan Metrik
os.makedirs(os.path.join("..", "outputs", "metrics"), exist_ok=True)
os.makedirs(os.path.join("..", "outputs", "models"), exist_ok=True)

metrics_results = {
    "method": "GridSearchCV",
    "execution_time_seconds": execution_time,
    "best_params": best_params,
    "best_validation_mse": float(best_val_mse),
    "test_mse": float(test_mse),
    "test_rmse": float(test_rmse),
    "mape": float(mape)
}

with open(os.path.join("..", "outputs", "metrics", "gridsearch_results.json"), "w") as f:
    json.dump(metrics_results, f, indent=4)

import joblib
joblib.dump(best_model, (os.path.join("..", "outputs", "models", "xgboost_gridsearch.pkl")))
print("Model dan metrik berhasil disimpan.")


### Visualisasi: Actual vs Predicted

In [ ]:
plot_dir = os.path.join("..", "outputs", "plots")
os.makedirs(plot_dir, exist_ok=True)

plt.figure(figsize=(14, 6))
plt.plot(y_test.index, y_test.values, label="Actual IHSG Price", color='#1f77b4', linewidth=1.5)
plt.plot(y_test.index, test_preds, label="Predicted", color='#ff7f0e', linestyle='--', linewidth=1.5)
plt.title("IHSG Closing Price Prediction Results on Test Set")
plt.xlabel("Date")
plt.ylabel("Index Value")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "actual_vs_predicted_gridsearch.png"), dpi=300)
plt.show()


### Visualisasi: Feature Importance

In [ ]:
importance_scores = best_model.feature_importances_
importance_df = pd.DataFrame({
    'Features': X_train.columns,
    'Importance': importance_scores
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Features', data=importance_df, palette="viridis")
plt.title("Feature Importance Scores")
plt.tight_layout()
plt.savefig(os.path.join(plot_dir, "feature_importance_gridsearch.png"), dpi=300)
plt.show()
